In [55]:
#RNN for sentiment analysis.
import pandas as pd

In [56]:
df = pd.read_csv("IMDB Dataset.csv")

In [57]:
df.shape

(50000, 2)

In [58]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [59]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [60]:
df.drop_duplicates(inplace=True)

In [61]:
df.shape

(49582, 2)

# Text Pre-processing

In [62]:
# 1)Converting to lower-case
# 2) remove url
# 3) remove html tags
# 4) remove punctuations (,,?, # etc)
# 5) remove stopwords via existing libraries.
# 6) stemming/basic form via existing libraries.
# 7) encode sentiment values 
# 8) vectorization :- TF-IDF

In [63]:
#Converting to lower case.
df["review"] = df["review"].str.lower()

In [64]:
import re

sample_text = "abc is the word, abc" #abc => xyz

new_text = re.sub("abc", "xyz", sample_text)

In [65]:
new_text

'xyz is the word, xyz'

In [66]:
#remove urls
def remove_urls(text):
    new_text = re.sub(r"http\S+", "", text)  #(pattern, replacement, string) 
    #this pattern is used for removal of all links
    #replacement empty space ke saath substitute kardega.
    return new_text


df["review"] = df["review"].apply(remove_urls)

In [67]:
#Remove punctuations

def remove_punc(text):
    new_text = re.sub(r"[^A-Za-z0-9\s]", "", text) #punctuations ke liye regex mein different pattern apply hoga bas for removal.
    return new_text

df["review"] = df["review"].apply(remove_punc) #.apply automatically calls the func for every row of df["review"]

In [68]:
#remove html tags

def remove_html(text):
    new_text = re.sub(r"<.*?>", "", text)
    return new_text

df["review"] = df["review"].apply(remove_html)

In [69]:
!pip install nltk

In [70]:
#removing stopwords
#to remove stopwords you first need to tokenize.
#nltk helps in removing stopwords and tokenization.
import nltk

nltk.download("punkt") #official tokenizer in nltk
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /Users/apple/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/apple/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/apple/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [71]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [72]:
sample_Text = "I like coding in python!"

tokens = word_tokenize(sample_Text)

In [73]:
tokens

['I', 'like', 'coding', 'in', 'python', '!']

In [74]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english") #means english ke stopwords.
    #in tokens ke andar hum check karenge ke hamara koi stop_word exist to nahi karta.
    #agar karta hai to remove karna hai.
    for word in tokens:
        if word in stop_words:
            new_text = text.replace(word, "")

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [75]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [76]:
#stemming

In [79]:
# running -> run
# played -> play
# PoterStemming

from nltk.stem import PorterStemmer

In [80]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for word in tokens:
        stemmed_token = ps.stem(word)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [81]:
# encoding for target value

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [82]:
Y = df["sentiment"]

In [ ]:
# Vectorization
#covert text data to numbers.

#sklearn.TfidfVectorizer
#it converts a collection of raw documents to a matrix of TF-IDF features.

In [83]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [85]:
tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

In [86]:
X #Converted to numbers

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5875056 stored elements and shape (49582, 5000)>

# Dataset & DataLoaders

In [109]:
from sklearn.model_selection import train_test_split

In [110]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

In [111]:
X_train.shape

(39665, 5000)

In [112]:
X_test.shape

(9917, 5000)

In [113]:
import torch
import torch.nn as nn

In [114]:
#X_train and X_test are now sparse matrix
#we do not use sparse matrix directly in dataset and dataloader.

In [115]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [116]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train.values, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
Y_test_tensor = torch.tensor(Y_test.values, dtype=torch.float32)

In [117]:
from torch.utils.data import TensorDataset, DataLoader

#train_dataset = TensorDataset(in_features, out_features)
train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
#test_dataset = TensorDataset(in_features, out_features)
test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)

In [118]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=True)

# Build RNN

In [119]:
import torch.optim as optim

In [120]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1) #1 because its many to one. #only one output.

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size) #initialize h0

        out, _ = self.rnn(x, h0) # returns 2 values. 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [121]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss() #Binary cross entropy loss for binary classification.
optimizer = optim.Adam(model.parameters())

In [122]:
#unsqueeze and squeeze are used to change dimension.
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) # (batch_size, 1) #raw outputs/score.

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.20091529190540314
epoch = 2/10 and loss = 0.11937906593084335
epoch = 3/10 and loss = 0.2197277843952179
epoch = 4/10 and loss = 0.15382130444049835
epoch = 5/10 and loss = 0.2895295321941376
epoch = 6/10 and loss = 0.346570760011673
epoch = 7/10 and loss = 0.11955131590366364
epoch = 8/10 and loss = 0.28088128566741943
epoch = 9/10 and loss = 0.14159274101257324
epoch = 10/10 and loss = 0.17754317820072174


In [123]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 87.49621861450035
